# Agent Loop 初探

一个 Agent Loop 需要包括

- 任务描述

// loop starts

- 模型思考

- 模型工具调用

- 受约束的执行 

- 观察结果

// loop ends

- 任务结束

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from openai import OpenAI

for key in ('HTTP_PROXY', 'HTTPS_PROXY', 'http_proxy', 'https_proxy'):
    os.environ.pop(key, None)
os.environ['NO_PROXY'] = '*'

def load_env(path='.env'):
    config = {}
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        key, _, value = line.partition('=')
        config[key.strip()] = value.strip().strip('\'"')
    return config

env = load_env()
MODEL = env['OPENAI_MODEL']
client = OpenAI(
    api_key=env['OPENAI_API_KEY'],
    base_url=env['OPENAI_BASE_URL'],
    timeout=120.0,
)
print('Model:', MODEL)

: 

In [2]:
print('Base URL:', env['OPENAI_BASE_URL'])

Base URL: https://api.ubec.top/v1


In [ ]:
import requests

resp = requests.post(
    f'{env["OPENAI_BASE_URL"]}/chat/completions',
    headers={
        'Authorization': f'Bearer {env["OPENAI_API_KEY"]}',
        'Content-Type': 'application/json',
    },
    json={
        'model': MODEL,
        'messages': [{'role': 'user', 'content': 'Say "pong" and nothing else.'}],
        'max_tokens': 10,
    },
    timeout=30,
)
print('Status:', resp.status_code)
print('Response:', json.dumps(resp.json()))

: 

## 一个有 bug 的小工作区

沙盒起始时有一个「问候语格式化」的 bug，让模型几个回合就能走完一遍「读 -> 改 -> 测」。

In [ ]:
LAB = Path.cwd() / '_agent_loop_lab'

def reset_lab():
    LAB.mkdir(exist_ok=True)
    (LAB / 'app.py').write_text("def greet(name):\n    return f'Hello {name}!'\n")
    (LAB / 'test_app.py').write_text(
        "from app import greet\n"
        "assert greet('Ada') == 'Hello, Ada!'\n"
        "print('ok')\n"
    )

reset_lab()
print('Workspace:', LAB)

: 

## 声明工具

模型不能跑任意 shell，只有三个工具：读文件、写文件、跑测试。`TOOLS` 是给模型看的 schema，`execute_tool` 是真正执行的地方。

In [ ]:
TOOLS = [
    {'type': 'function', 'function': {'name': 'read_file', 'description': '读 lab 里的一个文件',
     'parameters': {'type': 'object', 'properties': {'path': {'type': 'string'}}, 'required': ['path']}}},
    {'type': 'function', 'function': {'name': 'write_file', 'description': '写 lab 里的一个文件',
     'parameters': {'type': 'object', 'properties': {'path': {'type': 'string'}, 'content': {'type': 'string'}}, 'required': ['path', 'content']}}},
    {'type': 'function', 'function': {'name': 'run_tests', 'description': '跑 lab 里的测试',
     'parameters': {'type': 'object', 'properties': {}}}},
]


def execute_tool(name, args):
    if name == 'read_file':
        return (LAB / args['path']).read_text()
    if name == 'write_file':
        (LAB / args['path']).write_text(args['content'])
        return 'written'
    if name == 'run_tests':
        r = subprocess.run([sys.executable, 'test_app.py'], cwd=LAB, capture_output=True, text=True)
        return f'returncode={r.returncode}\nstdout={r.stdout}\nstderr={r.stderr}'
    return f'unknown tool: {name}'

: 

## 最小的循环

这就是整个 harness：把历史发给模型，模型要么给最终答复、要么请求调用工具；我们执行工具、把结果追加回历史，再问下一轮。要不要继续，由模型自己决定

In [9]:
INSTRUCTIONS = (
    'You are a careful coding agent inside a disposable teaching lab. '
    'Use only the declared tools. For the greeting task: read test_app.py, read app.py, '
    'make the smallest edit, then run the test. '
    'Never claim the test passed unless run_tests says so.'
)


def run_agent(task, max_steps=6):
    history = [
        {'role': 'system', 'content': INSTRUCTIONS},
        {'role': 'user', 'content': task},
    ]
    for step in range(1, max_steps + 1):
        resp = client.chat.completions.create(
            model=MODEL,
            messages=history,
            tools=TOOLS,
            parallel_tool_calls=False,
            timeout=120.0,
        )
        msg = resp.choices[0].message
        record = msg.model_dump()
        reasoning = record.pop('reasoning_content', None)   # 思考过程只展示，不回填历史
        history.append(record)                              # 把模型这轮的输出追加回历史
        calls = msg.tool_calls or []

        print(f'\n=== step {step} ===')
        if reasoning:
            print('THINK:', reasoning)
        if msg.content:
            print('MODEL:', msg.content)
        if not calls:                                       # 没有工具调用 = 任务结束
            return msg.content

        for call in calls:
            args = json.loads(call.function.arguments)
            result = execute_tool(call.function.name, args)
            print(f'TOOL  {call.function.name}({args}) -> {result!r}')
            history.append({'role': 'tool', 'tool_call_id': call.id, 'content': result})

    print('Stopped after max_steps')

In [10]:
reset_lab()
run_agent('Fix the greeting formatting bug and run the test.')
print('\napp.py now:\n' + (LAB / 'app.py').read_text())


=== step 1 ===
THINK: Let me start by reading the relevant files to understand the current state of the code.
TOOL  read_file({'path': 'app.py'}) -> "def greet(name):\n    return f'Hello {name}!'\n"
TOOL  read_file({'path': 'test_app.py'}) -> "from app import greet\nassert greet('Ada') == 'Hello, Ada!'\nprint('ok')\n"

=== step 2 ===
THINK: The test expects `greet('Ada')` to return `'Hello, Ada!'` (with a comma after "Hello"), but the current implementation returns `'Hello Ada!'` (without the comma).

I need to add a comma after "Hello" in the greet function.
TOOL  write_file({'path': 'app.py', 'content': "def greet(name):\n    return f'Hello, {name}!'\n"}) -> 'written'

=== step 3 ===
THINK: Now let me run the test to verify it passes.
TOOL  run_tests({}) -> 'returncode=0\nstdout=ok\n\nstderr='

=== step 4 ===
THINK: The test passed.
MODEL: The fix was straightforward: the greeting was missing a comma after "Hello". I changed `f'Hello {name}!'` to `f'Hello, {name}!'` and the test pas